# Will FC Thun win Swiss Super League in Season 2025/2026?

In [60]:
import polars as pl

In [61]:
cfg = pl.Config()
cfg.set_tbl_rows(20)

polars.config.Config

In [62]:
# Load data https://www.football-data.co.uk/switzerland.php
raw_df = pl.read_csv(source="data/SWZ.csv", try_parse_dates=True)

In [63]:
raw_df.head(5)

Country,League,Season,Date,Time,Home,Away,HG,AG,Res,PSCH,PSCD,PSCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365CH,B365CD,B365CA
str,str,str,date,time,str,str,i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str
"""Switzerland""","""Super League""","""2012/2013""",2012-07-13,18:45:00,"""Servette""","""Basel""",0,1,"""A""",6.22,4.17,1.6,6.22,4.2,1.68,5.32,3.85,1.59,null,null,null,null,null,null
"""Switzerland""","""Super League""","""2012/2013""",2012-07-14,18:45:00,"""Thun""","""Lausanne""",0,0,"""D""",1.75,3.8,5.23,1.77,3.81,5.38,1.7,3.51,4.85,null,null,null,null,null,null
"""Switzerland""","""Super League""","""2012/2013""",2012-07-15,12:45:00,"""Grasshoppers""","""Sion""",0,2,"""A""",3.27,3.36,2.39,3.35,3.41,2.41,3.07,3.24,2.25,null,null,null,null,null,null
"""Switzerland""","""Super League""","""2012/2013""",2012-07-15,12:45:00,"""Luzern""","""Zurich""",1,1,"""D""",2.04,3.65,3.89,2.2,3.8,3.89,2.1,3.33,3.31,null,null,null,null,null,null
"""Switzerland""","""Super League""","""2012/2013""",2012-07-15,15:00:00,"""St. Gallen""","""Young Boys""",1,1,"""D""",3.62,3.67,2.12,3.69,3.67,2.15,3.32,3.43,2.06,null,null,null,null,null,null


In [64]:
# Keep needed columns and filter for Super League
raw_df = raw_df.select(["League", "Season", "Date", "Home", "Away", "HG", "AG", "Res"])
raw_df = raw_df.filter(pl.col("League") == "Super League")

In [65]:
# Add points for home and away team
raw_df = raw_df.with_columns(
    pl.col("Res").replace_strict({"H": 3, "A": 0}, default=1).alias("PointsH"),
    pl.col("Res").replace_strict({"H": 0, "A": 3}, default=1).alias("PointsA")
).with_row_index("MatchId")

In [66]:
# Construct evolution of table
table_evolution = (
    raw_df
        .with_columns(
            pl.col("Date").cast(pl.Date)
        )
        .select(
            "MatchId", "Season", "Date",
            pl.col("Home").alias("Team"),
            pl.col("HG").alias("Goals"),
            pl.col("AG").alias("GoalsAgainst"),
            pl.col("PointsH").alias("Points"),
        )
        .vstack(
            raw_df.select(
                "MatchId", "Season", "Date",
                pl.col("Away").alias("Team"),
                pl.col("AG").alias("Goals"),
                pl.col("HG").alias("GoalsAgainst"),
                pl.col("PointsA").alias("Points"),
            )
        )
        .sort(["Season", "Team", "Date", "MatchId"])
)

In [67]:
# Cumulative stats per team and season
table_evolution = table_evolution.with_columns(
    pl.col("Points").cum_sum().over(["Season", "Team"]).alias("CumPoints"),
    pl.col("Goals").cum_sum().over(["Season", "Team"]).alias("CumGoals"),
    pl.col("GoalsAgainst").cum_sum().over(["Season", "Team"]).alias("CumGoalsAgainst"),
    (pl.col("Goals") - pl.col("GoalsAgainst")).cum_sum().over(["Season", "Team"]).alias("CumGoalsDifference")
)

In [68]:
# Add matchday index per team and season
table_evolution = table_evolution.with_columns(
    pl.arange(1, pl.len() + 1)
        .over(["Season", "Team"])
        .alias("Matchday")
)

In [69]:
# Relegation games are also included - we want to exclude them. 
# 36 rounds used to be played until 2022/2023. Since then 38 rounds are played.
table_evolution = table_evolution.filter(
    pl.when(pl.col("Season") <= "2022/2023")
        .then(pl.col("Matchday") <= 36)
        .otherwise(pl.col("Matchday") <= 38)
    )

In [70]:
table_evolution.head()

MatchId,Season,Date,Team,Goals,GoalsAgainst,Points,CumPoints,CumGoals,CumGoalsAgainst,CumGoalsDifference,Matchday
u32,str,date,str,i64,i64,i64,i64,i64,i64,i64,i64
0,"""2012/2013""",2012-07-13,"""Basel""",1,0,3,3,1,0,1,1
5,"""2012/2013""",2012-07-21,"""Basel""",2,2,1,4,3,2,1,2
10,"""2012/2013""",2012-07-28,"""Basel""",2,2,1,5,5,4,1,3
15,"""2012/2013""",2012-08-04,"""Basel""",1,1,1,6,6,5,1,4
24,"""2012/2013""",2012-08-12,"""Basel""",3,1,3,9,9,6,3,5


## Reconstruct Table by Season and Matchday

In [71]:
def get_table_for_season_matchday(df: pl.DataFrame, season: str, matchday: int) -> pl.DataFrame:
    # If matchday == -1 -> get last available match day
    if matchday == -1:
        last_md = (
            df
            .filter(pl.col("Season") == season)
            .select(pl.col("Matchday").max())
            .item()
        )
        matchday = int(last_md)

    return (
        df
        .filter(
            (pl.col("Season") == season) &
            (pl.col("Matchday") == matchday)
        )
        .sort(["CumPoints", "CumGoals"], descending=True)
        .select("Season", "Team", "CumPoints", "CumGoals", "CumGoalsAgainst", "CumGoalsDifference", "Matchday")
        .rename({"CumPoints": "Points", "CumGoals": "G", "CumGoalsAgainst": "GA", 
                 "CumGoalsDifference": "GD", "Matchday": "Round"})
    )

get_table_for_season_matchday(df=table_evolution, season="2024/2025", matchday=-1)

Season,Team,Points,G,GA,GD,Round
str,str,i64,i64,i64,i64,i64
"""2024/2025""","""Basel""",73,91,43,48,38
"""2024/2025""","""Servette""",63,64,55,9,38
"""2024/2025""","""Young Boys""",61,60,49,11,38
"""2024/2025""","""Lugano""",54,55,58,-3,38
"""2024/2025""","""Lausanne""",53,62,54,8,38
"""2024/2025""","""Zurich""",53,56,57,-1,38
"""2024/2025""","""Luzern""",52,66,64,2,38
"""2024/2025""","""St. Gallen""",52,52,53,-1,38
"""2024/2025""","""Sion""",44,47,57,-10,38


## How is FC Thun performing compared to the champions since 2012/2013

In [56]:
# Add column with winner of each season
champions = (
    table_evolution
    .filter(
        (pl.col("Season") != "2025/2026") &
        (pl.col("Matchday") == pl.col("Matchday").max().over("Season"))
    )
    .group_by("Season")
    .agg(
        pl.col("Team").filter(pl.col("CumPoints") == pl.col("CumPoints").max()).first().alias("Winner")
    )
)

table_evolution = table_evolution.join(champions, on="Season", how="left").with_columns(
    (pl.col("Team") == pl.col("Winner")).alias("Winner")
)


In [57]:
# Calculate the stats of winners 
winners_stats = (
    table_evolution
    .filter(
        (pl.col("Winner") == True) &
        (pl.col("Season") != "2025/2026")
    )
    .group_by("Matchday")
    .agg([
        pl.col("CumPoints").mean().alias("mean_CumPoints"),
        pl.col("CumPoints").std().alias("std_CumPoints"),
        pl.col("CumGoals").mean().alias("mean_CumGoals"),
        pl.col("CumGoals").std().alias("std_CumGoals"),
        pl.col("CumGoalsAgainst").mean().alias("mean_CumGoalsAgainst"),
        pl.col("CumGoalsAgainst").std().alias("std_CumGoalsAgainst"),
    ])
    .sort("Matchday")
)


In [105]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"], 
    y=winners_stats["mean_CumPoints"],
    mode='lines+markers',
    line_color="black",
    name='Avg Points of Champion'
))

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"],
    y=winners_stats["mean_CumPoints"] + winners_stats["std_CumPoints"],
    mode='lines',
    line_color="black",
    line_width=0,
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"],
    y=winners_stats["mean_CumPoints"] - winners_stats["std_CumPoints"],
    fill='tonexty',
    fillcolor='rgba(0,0,0,0.2)',
    line_color="black",
    line_width=0,
    name='±1 Std Dev'
))

team_colors = {
    "Thun": "#e2001a",        
    "St. Gallen": "#2c8548",  
    "Young Boys": "#f9cc11",  
}

for team in ["Thun", "St. Gallen", "Young Boys"]:
    latestTeam = table_evolution.filter(
        (pl.col("Team") == team) & 
        (pl.col("Season") == "2025/2026")
    )
    
    fig.add_trace(go.Scatter(
        x=latestTeam["Matchday"], 
        y=latestTeam["CumPoints"],
        mode='lines+markers',
        name=team,
        line=dict(color=team_colors[team], width=3),
        marker=dict(color=team_colors[team], size=8)
    ))

fig.update_layout(
    title={
        'text': 'Super League 2025/2026 vs. Historical Champions - Points',
        'x': 0.5,
        'font': {'size': 20, 'color': 'black'}
    },
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.9)',
        font=dict(color='black')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        title="Matchday",
        tickfont=dict(color='black'),
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=1,
        range=[0, 35]
    ),
    yaxis=dict(
        title="Cumulative Points",
        tickfont=dict(color='black'),
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=1,
        range=[0, max(winners_stats["mean_CumPoints"]) * 1.1]
    ),
    height=500,
    width=700,
    hovermode='x unified',
    font=dict(color='black')
)

fig.show()


In [104]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"], 
    y=winners_stats["mean_CumGoals"],
    mode='lines+markers',
    line_color="black",
    name='Avg Goals of Champion'
))

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"],
    y=winners_stats["mean_CumGoals"] + winners_stats["std_CumGoals"],
    mode='lines',
    line_color="black",
    line_width=0,
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"],
    y=winners_stats["mean_CumGoals"] - winners_stats["std_CumGoals"],
    fill='tonexty',
    fillcolor='rgba(0,0,0,0.2)',
    line_color="black",
    line_width=0,
    name='±1 Std Dev'
))

team_colors = {
    "Thun": "#e2001a",        
    "St. Gallen": "#2c8548",  
    "Young Boys": "#f9cc11",  
}

for team in ["Thun", "St. Gallen", "Young Boys"]:
    latestTeam = table_evolution.filter(
        (pl.col("Team") == team) & 
        (pl.col("Season") == "2025/2026")
    )
    
    fig.add_trace(go.Scatter(
        x=latestTeam["Matchday"], 
        y=latestTeam["CumGoals"],
        mode='lines+markers',
        name=team,
        line=dict(color=team_colors[team], width=3),
        marker=dict(color=team_colors[team], size=8)
    ))

fig.update_layout(
    title={
        'text': 'Super League 2025/2026 vs. Historical Champions - Goals',
        'x': 0.5,
        'font': {'size': 20, 'color': 'black'}
    },
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.9)',
        font=dict(color='black')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        title="Matchday",
        tickfont=dict(color='black'),
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=1,
        range=[0, 35]
    ),
    yaxis=dict(
        title="Cumulative Goals",
        tickfont=dict(color='black'),
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=1,
        range=[0, max(winners_stats["mean_CumGoals"]) * 1.1]
    ),
    height=500,
    width=700,
    hovermode='x unified',
    font=dict(color='black')
)

fig.show()


In [106]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"], 
    y=winners_stats["mean_CumGoalsAgainst"],
    mode='lines+markers',
    line_color="black",
    name='Avg Goals Against of Champion'
))

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"],
    y=winners_stats["mean_CumGoalsAgainst"] + winners_stats["std_CumGoalsAgainst"],
    mode='lines',
    line_color="black",
    line_width=0,
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=winners_stats["Matchday"],
    y=winners_stats["mean_CumGoalsAgainst"] - winners_stats["std_CumGoalsAgainst"],
    fill='tonexty',
    fillcolor='rgba(0,0,0,0.2)',
    line_color="black",
    line_width=0,
    name='±1 Std Dev'
))

team_colors = {
    "Thun": "#e2001a",        
    "St. Gallen": "#2c8548",  
    "Young Boys": "#f9cc11",  
}

for team in ["Thun", "St. Gallen", "Young Boys"]:
    latestTeam = table_evolution.filter(
        (pl.col("Team") == team) & 
        (pl.col("Season") == "2025/2026")
    )
    
    fig.add_trace(go.Scatter(
        x=latestTeam["Matchday"], 
        y=latestTeam["CumGoalsAgainst"],
        mode='lines+markers',
        name=team,
        line=dict(color=team_colors[team], width=3),
        marker=dict(color=team_colors[team], size=8)
    ))

fig.update_layout(
    title={
        'text': 'Super League 2025/2026 vs. Historical Champions - Goals Against',
        'x': 0.5,
        'font': {'size': 20, 'color': 'black'}
    },
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.9)',
        font=dict(color='black')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        title="Matchday",
        tickfont=dict(color='black'),
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=1,
        range=[0, 35]
    ),
    yaxis=dict(
        title="Cumulative Goals Against",
        tickfont=dict(color='black'),
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=1,
        range=[0, max(winners_stats["mean_CumGoalsAgainst"]) * 1.1]
    ),
    height=500,
    width=700,
    hovermode='x unified',
    font=dict(color='black')
)

fig.show()
